In [6]:
"""
候选因子：下午 VWAP / 收盘价压力反转
============================================================
参考已提交成功文件：factor_vwap_am_pm.py

逻辑：
    保留完整时间戳，用 pandas 提取小时。若下午成交均价高于收盘价，
    表示下午冲高后收弱或尾盘有卖压，短期存在反转修复可能。

方向：
    因子值越大 → 预期未来收益越高。
"""

from __future__ import annotations


def main(datasources, start_date, end_date):
    import dai
    import pandas as pd
    import numpy as np

    table_name = datasources["bar1m"]

    sql = f"""
        SELECT
            date AS date,
            instrument,
            close,
            volume
        FROM {table_name}
    """

    raw = dai.query(
        sql,
        filters={"date": [start_date, end_date]},
        compression=True,
    ).df()

    raw["datetime"] = pd.to_datetime(raw["date"])
    raw["hour"] = raw["datetime"].dt.hour
    raw["minute"] = raw["datatime"].dt.minute
    raw["date"] = raw["datetime"].dt.normalize()

    is_afternoon = (raw["hour"] >= 13)& (raw["hour"] < 15)
    is_close_window = ((raw["hour"] == 14)&(raw["minute"] >= 30)) | (raw["hour"] == 15)
    is_morning = (raw["hour"] >= 9) & (raw["hour"] < 12)

    raw["close_window_amt"] = np.where(is_close_window, raw["close"] * raw["volume"], 0.0)
    raw["close_window_vol"] = np.where(is_close_window, raw["volume"], 0)

    raw["afternoon_amt"] = np.where(is_afternoon, raw["close"] * raw["volume"], 0.0)
    raw["afternoon_vol"] = np.where(is_afternoon, raw["volume"], 0)

    raw["morning_amt"] = np.where(is_morning, raw["close"] * raw["volume"], 0.0)
    raw["morning_vol"] = np.where(is_morning, raw["volume"], 0)

    daily = raw.groupby(["date", "instrument"]).agg(
        close_window_amt=("close_window_amt", "sum"),
        close_window_vol=("close_window_vol", "sum"),
        afternoon_amt=("afternoon_amt", "sum"),
        afternoon_vol=("afternoon_vol", "sum"),
        morning_amt=("morning_amt", "sum"),
        morning_vol=("morning_vol", "sum"),
        total_vol=("volume", "sum"),
        close_price=("close", "last"),
        open_price=("close", "first"),
    ).reset_index()

    daily["vwap_close_window"] = np.where(
        daily["close_window_vol"] > 0,
        daily["close_window_amt"] / daily["close_window_vol"],
        np.nan,
    )

    daily["vwap_pm"] = np.where(
        daily["afternoon_vol"] > 0,
        daily["afternoon_amt"] / daily["afternoon_vol"],
        np.nan,
    )

    daily["vwap_am"] = np.where(
        daily["morning_vol"] > 0,
        daily["morning_amt"] / daily["morning_vol"],
        np.nan,
    )

    daily["pm_vol_ratio"] = np.where(
        daily["total_vol"] > 0,
        daily["afternoon_vol"] / daily["total_vol"],
        0.0,
    )

    daily["close_window_ratio"] = np.where(
        daily["total_vol"] > 0,
        daily["close_window_vol"] / daily["total_vol"],
        0.0,
    )

    daily["price_range"] = np.where(
        daily["open_price"] > 0,
        (daily["vwap_pm"] - daily["open_price"]) / daily["open_price"],
        np.nan,
    )

    daily["pm_am_spread"] = np.where(
        daily["vwap_am"] > 0,
        (daily["vwap_pm"] - daily["vwap_am"]) / daily["vwap_am"],
        np.nan,
    )

    daily["raw_factor"] = (
        (daily["vwap_close_window"] / daily["close_price"])
        * (1.0 + daily["close_window_ratio"])
        * np.exp(daily["pm_am_spread"].fillna(0) * 10)
        * (1.0 + 0.8 * daily["pm_vol_ratio"])
    )

    daily["raw_factor"] = daily["raw_factor"].replace([np.inf, -np.inf], np.nan)
    daily = daily.dropna(subset=["raw_factor"])

    def normalize(group):
        group["factor"] = group["raw_factor"].rank(pct=True)
        group["factor"] = (group["factor"] - 0.5) * 2
        return group

    daily = daily.groupby("date", group_keys=False).apply(normalize)
    daily["factor"] = pd.to_numeric(daily["factor"], errors="coerce")

    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
    ).df()
    stk_pool["date"] = pd.to_datetime(stk_pool["date"]).dt.normalize()

    return (
        pd.merge(daily, stk_pool, how="inner", on=["date", "instrument"])
        .dropna(subset=["factor"])
        .sort_values(["date", "instrument"])
        .reset_index(drop=True)
        .loc[:, ["date", "instrument", "factor"]]
    )